# Ganadero.org — Apache Spark Livestock Analytics Lab
**Standalone teaching notebook | Sessions 12 + 13 | PySpark**

Explore a fictional Spanish livestock management dataset inspired by Ganadero.org. All farms, animals, events and measurements are **synthetic**, not real customer records.

**Learning goals:** read multiple CSV sources, inspect and enforce schemas, transform columns, filter records, join datasets, aggregate metrics, use windows, and write Parquet.

**Files:** `farms.csv`, `animals.csv`, `weights.csv`, `health_events.csv`, `births.csv`. Place this notebook in the same folder as the CSV files. Requires Java and `pyspark` (`pip install pyspark`). No external services required.

## 1. Start Spark and locate the data

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder.appName("GanaderoTeachingLab")
         .master("local[*]").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
DATA_DIR = Path.cwd()
assert (DATA_DIR / "animals.csv").exists(), "Run the notebook from the extracted Ganadero_Spark_Lab folder"
print("Spark:", spark.version, "| Data:", DATA_DIR)

## 2. Data Sources API — CSV and explicit schemas

CSV does not inherently preserve Spark data types. Define schemas rather than relying on inference.

In [ ]:
animal_schema = T.StructType([
    T.StructField("animal_id", T.StringType(), False),
    T.StructField("farm_id", T.StringType(), False),
    T.StructField("animal_name", T.StringType(), True),
    T.StructField("sex", T.StringType(), True),
    T.StructField("breed", T.StringType(), True),
    T.StructField("birth_date", T.DateType(), True),
    T.StructField("status", T.StringType(), True),
])
animals = (spark.read.option("header", True).option("dateFormat", "yyyy-MM-dd")
           .schema(animal_schema).csv(str(DATA_DIR / "animals.csv")))
farms = spark.read.option("header", True).csv(str(DATA_DIR / "farms.csv"))
weights = (spark.read.option("header", True).option("inferSchema", True)
           .csv(str(DATA_DIR / "weights.csv"))
           .withColumn("measurement_date", F.to_date("measurement_date")))
health = (spark.read.option("header", True).option("inferSchema", True)
          .csv(str(DATA_DIR / "health_events.csv"))
          .withColumn("event_date", F.to_date("event_date")))
births = (spark.read.option("header", True).option("inferSchema", True)
          .csv(str(DATA_DIR / "births.csv"))
          .withColumn("birth_date", F.to_date("birth_date")))
animals.printSchema()
animals.show(5, truncate=False)
print("Animals:", animals.count(), "Farms:", farms.count())

## 3. Columns, expressions, filtering and casting

Use a fixed analysis date to make classroom results reproducible.

In [ ]:
analysis_date = F.to_date(F.lit("2026-11-24"))
active = (animals.filter(F.col("status") == "active")
          .withColumn("age_months", F.floor(F.months_between(analysis_date, "birth_date")))
          .withColumn("sex_label", F.when(F.col("sex") == "F", "Female").otherwise("Male")))
active.select("animal_id", "breed", "sex_label", "age_months").show(10)

## 4. GroupBy — herd composition by breed and sex

In [ ]:
(active.groupBy("breed", "sex_label")
 .agg(F.count("*").alias("animals"))
 .orderBy("breed", "sex_label")).show(30, truncate=False)

## 5. Join farms and animals — animals per farm

In [ ]:
farm_summary = (farms.join(active, "farm_id", "left")
    .groupBy("farm_id", "farm_name", "province")
    .agg(F.count("animal_id").alias("active_animals"),
         F.countDistinct("breed").alias("distinct_breeds"))
    .orderBy(F.desc("active_animals")))
farm_summary.show(truncate=False)

## 6. Window functions — latest weight per animal

Multiple measurements exist per animal. Select the most recent before calculating averages; otherwise animals with more measurements will be overrepresented.

In [ ]:
window = Window.partitionBy("animal_id").orderBy(F.col("measurement_date").desc(), F.col("weight_id").desc())
latest_weights = (weights.withColumn("rn", F.row_number().over(window))
                  .filter(F.col("rn") == 1).drop("rn"))
latest_weights.join(active, "animal_id", "inner").groupBy("breed").agg(
    F.count("*").alias("weighed_animals"),
    F.round(F.avg("weight_kg"), 1).alias("avg_latest_weight_kg")
).orderBy(F.desc("avg_latest_weight_kg")).show(truncate=False)

## 7. Health event analytics — costs by farm and event type

In [ ]:
(health.join(animals.select("animal_id", "farm_id"), "animal_id")
 .join(farms, "farm_id")
 .groupBy("farm_name", "event_type")
 .agg(F.count("*").alias("events"), F.round(F.sum("cost_eur"), 2).alias("total_cost_eur"))
 .orderBy("farm_name", "event_type")).show(50, truncate=False)

## 8. Reproduction analytics — recorded calves per farm

Births refer to the mother animal. This is a synthetic event log, not a full reproductive performance model.

In [ ]:
(births.join(animals.select(F.col("animal_id").alias("mother_id"), "farm_id"), "mother_id")
 .join(farms, "farm_id")
 .groupBy("farm_name")
 .agg(F.count("birth_id").alias("birth_events"), F.sum("calves_born").alias("calves_recorded"))
 .orderBy(F.desc("calves_recorded"))).show(truncate=False)

## 9. Write Parquet and read it back

Parquet preserves types and is well suited to analytical workloads.

In [ ]:
out = str(DATA_DIR / "output" / "active_animals_parquet")
active.write.mode("overwrite").parquet(out)
reloaded = spark.read.parquet(out)
print("Parquet records:", reloaded.count())
reloaded.printSchema()
assert reloaded.count() == active.count()

## 10. Student challenge — 15 minutes

**Task A:** Identify the three farms with the most active female cattle.

**Task B:** Find the average **latest** weight of active animals by farm, rounded to one decimal.

**Task C:** Find the number of animals with **no** recorded health event using a `left_anti` join.

**Task D (bonus):** Calculate each breed's share of the active herd as a percentage.

Try the tasks in the blank cell below **before** opening the solutions.

In [ ]:
# Your solutions here


## 11. Instructor solutions — reveal after the challenge

In [ ]:
# A: top three farms by active females
(active.filter(F.col("sex") == "F").join(farms, "farm_id")
 .groupBy("farm_name").count().orderBy(F.desc("count")).limit(3)).show()

# B: average latest weight by farm, active animals only
(latest_weights.join(active.select("animal_id", "farm_id"), "animal_id")
 .join(farms, "farm_id").groupBy("farm_name")
 .agg(F.round(F.avg("weight_kg"), 1).alias("avg_kg"))
 .orderBy("farm_name")).show()

# C: active animals with no health events
no_health = active.join(health.select("animal_id").distinct(), "animal_id", "left_anti")
print("Active animals without a health event:", no_health.count())

# D: breed percentages
n = active.count()
(active.groupBy("breed").count()
 .withColumn("pct", F.round(F.col("count") * 100 / F.lit(n), 1))
 .orderBy(F.desc("pct"))).show()

## 12. Quick checks and wrap-up

In [ ]:
assert animals.count() == 240
assert farms.count() == 6
assert animals.select("animal_id").distinct().count() == 240
assert weights.count() >= 480
assert active.count() <= animals.count()
assert latest_weights.select("animal_id").distinct().count() == latest_weights.count()
print("Dataset checks passed.")
# Optional: spark.stop()

### Discussion prompts
1. Why does a CSV require schema handling but Parquet usually does not?
2. What goes wrong if we average **all** historical weights instead of each animal's latest weight?
3. When would a `left_anti` join help identify missing records?
4. Which metrics should be treated as descriptive rather than veterinary advice?

**Next steps:** partition Parquet by farm, test null handling, add quality checks, or connect to PostgreSQL/MongoDB using appropriate connectors.